In [1]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

(Cookbook_SymmetryPoseRMSD)=
# Comparing ligand poses

*Preserving original positional RMSD while evaluating admitted chemical symmetry.*

For redocking you need query and reference coordinates in the same receptor frame. Register the atom population, original map and chemical policy before interpreting a recovery threshold. This synthetic control illustrates provider geometry; it does not reclassify any existing docking population.

:::{versionadded} 1.0.0
:::

## Declaring chemistry and poses

In [2]:
import molsysmt as msm
import numpy as np
from rdkit import Chem

molsys = msm.convert(Chem.MolFromSmiles('c1ccccc1'), to_form='molsysmt.MolSys')
phase = np.arange(6) * np.pi / 3
xyz = 0.14 * np.column_stack((np.cos(phase), np.sin(phase), np.zeros(6)))
molsys.structures.append(coordinates=msm.pyunitwizard.quantity([xyz, xyz[[1, 2, 3, 4, 5, 0]], xyz + [0, 0, 0.2]], 'nm'))
original_map = np.column_stack((np.arange(6), np.arange(6)))
correspondences = msm.topology.get_atom_correspondences(molsys)
assert correspondences['search_complete']

## Evaluating both measures

Use one original map for the positional baseline and let chemical admission determine the corrected map. Requested structure order is preserved. The translated structure remains displaced.

In [3]:
result = msm.structure.get_symmetry_corrected_rmsd(molsys, selection='all', correspondences=correspondences, structure_indices=[2, 1, 0], baseline_atom_correspondence=original_map)
print('Raw:', result['raw_rmsd'])
print('Corrected:', result['rmsd'])
np.testing.assert_allclose(msm.pyunitwizard.get_value(result['rmsd'], to_unit='nm'), [0.2, 0, 0], atol=1e-14)
print('Chosen maps:', result['correspondence_indices'])

Raw: [0.2 0.14 0.0] nanometer
Corrected: [0.2 0.0 0.0] nanometer
Chosen maps: [0 3 0]


## Recording a conditional comparison

If atom identity is constrained by your experiment, declare partial anchors. If an upstream tool proposes complete maps, provide those maps explicitly. The chemical validator checks the complete component in both cases. `baseline_atom_correspondence` remains independent: it only defines the original positional baseline.

Here you intentionally limit the comparison to the original map. The rotated/permuted structure no longer attains the unrestricted zero minimum. Retain `search_scope` alongside the value when reporting or applying a pose threshold.


In [4]:
conditional = msm.structure.get_symmetry_corrected_rmsd(
    molsys, selection='all', structure_indices=[1],
    fixed_atom_correspondence=[[0, 0]],
    candidate_atom_correspondences=original_map)
assert conditional['search_scope'] == 'supplied_atom_correspondences'
np.testing.assert_allclose(msm.pyunitwizard.get_value(conditional['rmsd'], to_unit='nm'), [0.14])
print('Declared scope:', conditional['search_scope'])
print('Conditional RMSD:', conditional['rmsd'])


Declared scope: supplied_atom_correspondences
Conditional RMSD: [0.14] nanometer


## Retaining provenance

Store detached bibliography beside your evaluation data. Ackredit is optional; applications own the session. It records reached calculations without changing scientific output. Saved result dictionaries retain original producer versions and reading them credits no new calculation. H5MSM 0.5 can retain source chemistry/coordinates; these result dictionaries are not a new H5MSM domain.

In [5]:
import json
bibliography = json.loads(json.dumps(result['attribution']))
assert bibliography == result['attribution']
print([(item['title'], item['roles']) for item in bibliography['items']])

[('DockRMSD: an open-source tool for atom mapping and RMSD calculation of symmetric molecules through graph isomorphism', ['methodological_inspiration']), ('spyrmsd: symmetry-corrected RMSD calculations in Python', ['methodological_inspiration']), ('MolSysMT', ['executed_software']), ('RDKit: Open-source cheminformatics', ['executed_software'])]


## Interpreting the scope

The profile admits exact declared-state graph symmetries, including complete unselected chemical context. It does not normalize resonance, protonation or tautomers, infer missing chemistry, superpose the ligand or reconstruct periodic images. For benzamidinium +1, phenyl symmetry is admitted while the declared N-single and N-double/+1 sites stay distinct. A consumer owns thresholds and separately registered recovery reports.

:::{seealso}
:class: dropdown
{ref}`Tutorial_Get_atom_correspondences`, {ref}`Tutorial_Get_symmetry_corrected_rmsd`, and {ref}`user-foundations-native-world-classes-molsysmt-molsys` explain mapping, units and native state ownership.
:::